In [22]:
import yfinance as yf
import pandas as pd
import numpy as np


In [23]:
#Fetch AAPL historical data 
df=yf.download("AAPL", start="2024-01-01", end="2026-01-01")

[*********************100%***********************]  1 of 1 completed


In [24]:
#Flatten multi-level column index
if isinstance(df.columns, pd.MultiIndex):
    df.columns=df.columns.get_level_values(0)

#convert column names to lowercase for consistency
df.columns=df.columns.str.lower()


In [25]:
#Clean the data
#Drop duplicate rows based on the index (date)
df=df.loc[~df.index.duplicated(keep="first")].sort_index()

#Forward fill missing values(like holidays or short gaps) then backward fill any remaining missing values
df=df.ffill().bfill()

In [26]:
#calculate daily returns
df["daily_return"]=df["close"].pct_change()
#calculate log returns for volatility analysis
df["log_return"]=np.log(df["close"]/df["close"].shift(1))

In [27]:
import talib

#Ensure your index is sorted chronologically
df=df.sort_index()

#Extract the close prices as a numpy array for TA-Lib 
close_prices=df["close"].to_numpy()

#Calculate 50-day and 200-day simple moving averages 
df["sma_50"]=talib.SMA(close_prices, timeperiod=50)
df["sma_200"]=talib.SMA(close_prices, timeperiod=200)



In [28]:
import numpy as np
#Generate the golden cross and death cross signals
df["prev_sma_50"]=df["sma_50"].shift(1)
df["prev_sma_200"]=df["sma_200"].shift(1)
#Logic conditions
golden_cross=(df["sma_50"] > df["sma_200"]) & (df["prev_sma_50"] <= df["prev_sma_200"])
death_cross=(df["sma_50"] < df["sma_200"]) & (df["prev_sma_50"] >= df["prev_sma_200"])

#Map into standard vector signals
#where; 1=Buy(Golden cross), -1=sell(Death cross), 0=Hold
df["raw_signal"]=np.where(golden_cross, 1, np.where(death_cross, -1, 0))
#shift the signal to the next day to avoid lookahead bias
df["executed_signal"]=df["raw_signal"].shift(1).fillna(0)

#drop all lag Nans caused by the moving average calculations and signal shifting
cleaned_df=df.dropna().copy()


#View rows where crosses occurred
crosses=cleaned_df[cleaned_df["raw_signal"]!=0]
print("--- Crossover Events Identified ---")
print(crosses[["close", "sma_50", "sma_200", "raw_signal"]].tail())


#calculate daily standard deviation or volatility of log returns
daily_volatility=cleaned_df["log_return"].std()

#calculate annualized volatility
annualized_volatility=daily_volatility * np.sqrt(252)  #Assuming 252 trading days in a year

#convert to a readable percentage format
print("\n--- Volatility summary ---")
print(f"Daily volatility: {daily_volatility:.4f}")
print(f"Annualized Volatility: {annualized_volatility * 100:.2f}%")

--- Crossover Events Identified ---
Price            close      sma_50     sma_200  raw_signal
Date                                                      
2025-04-07  180.350708  226.748033  227.076329          -1
2025-09-15  235.828842  220.653870  220.285789           1

--- Volatility summary ---
Daily volatility: 0.0189
Annualized Volatility: 30.05%


In [29]:
import scipy.stats as stats
#calculate value at risk (VaR) at 95% confidence level
portfolio_value=10000
confidence_level=0.95

#Isolate distribution patterns
mean_return=cleaned_df["log_return"].mean()
std_return=cleaned_df["log_return"].std()

#Map the exact standard deviation threshhold point using standard normal percent point function 
z_score=stats.norm.ppf(confidence_level)
var_percent=-(mean_return - z_score * std_return)
var_dollar=var_percent * portfolio_value

print(f"1-Day 95% value at risk (VaR): ${var_dollar:.2f}")



1-Day 95% value at risk (VaR): $305.94


In [30]:
import vectorbt as vbt
#map out clean Boolean signal execution masks
#True signals indicate the moment an order transaction is executed, while False indicates no action
vbt_close=cleaned_df["close"].squeeze().astype(float)
vbt_open=cleaned_df["open"].squeeze().astype(float)
vbt_entries=(cleaned_df["executed_signal"].squeeze()==1).to_numpy().astype(bool)
vbt_exits=(cleaned_df["executed_signal"].squeeze()== -1). to_numpy().astype(bool)

#Initialize the high-speed vectorbt simulation engine
#fees=0.05%(0.0005) broker commission, slippage=0.05% price penalty
pf=vbt.Portfolio.from_signals(
    close=vbt_close, #Realized tracking array
    open=vbt_open,   #Provided to execute trades accurately
    price=vbt_open,              #Executes at next-day open instead of close
    entries=vbt_entries,     #scaled entry points
    exits=vbt_exits,        #scaled exit points
    init_cash=10000,           #Base allocation balance
    fees=0.0005,               #0.05% broker fee
    slippage=0.0005,           #0.05% price penalty buffer
    sl_stop=0.03,               #Hard stop loss protection at 3% below entry price
    sl_trail=True,             #Transforms standard stop loss into a trailing stop loss
    tp_stop=0.06,             #Take profit threshold layer at 6% above entry price      
    freq="D"                  #Evaluates daily frequency on daily boundaries
)

#Extract comprehensive Analytics engine report
stats_report=pf.stats()

print("--- VectorBT complete strategy performance report---")
print(f"Total Portfolio Period:    {stats_report['Period']}")
print(f"Initial Deposited Capital: ${stats_report['Start Value']:,.2f}")
print(f"Final Strategy Value Net:  ${stats_report['End Value']:,.2f}")
print(f"Strategy Cumulative Return: {stats_report['Total Return [%]']:.2f}%")
print(f"Benchmark (Buy & Hold) Ret: {stats_report['Benchmark Return [%]']:.2f}%")
print(f"Max Strategy Drawdown:     {stats_report['Max Drawdown [%]']:.2f}%")
print(f"Total Trade Frictions Paid:${stats_report['Total Fees Paid']:,.2f}")
print(f"Strategy Win Rate:         {stats_report['Win Rate [%]']:.2f}%")
print(f"Calculated Sharpe Ratio:   {stats_report['Sharpe Ratio']:.4f}")



--- VectorBT complete strategy performance report---
Total Portfolio Period:    302 days 00:00:00
Initial Deposited Capital: $10,000.00
Final Strategy Value Net:  $10,627.40
Strategy Cumulative Return: 6.27%
Benchmark (Buy & Hold) Ret: 17.76%
Max Strategy Drawdown:     0.46%
Total Trade Frictions Paid:$10.31
Strategy Win Rate:         100.00%
Calculated Sharpe Ratio:   1.5856


In [31]:
# ==========================================
# ADVANCED RISK: HISTORICAL SIMULATION VAR
# ==========================================

# 1. Pull the true realistic daily returns from the VectorBT simulation engine
strategy_daily_returns = pf.returns()

# 2. Calculate the 5th percentile (95% Confidence Level) empirical floor boundary
confidence_level = 0.95
historical_var_pct = np.percentile(strategy_daily_returns, (1 - confidence_level) * 100)

# 3. Convert percentage threshold loss risk into a fixed dollar metric
portfolio_value = 10000.0
historical_var_dollar = -historical_var_pct * portfolio_value

print("--- Empirical Risk Engine Update ---")
print(f"95% 1-Day Historical Sim VaR (%): {historical_var_pct * 100:.4f}%")
print(f"95% 1-Day Historical Sim VaR ($): ${historical_var_dollar:.2f}")
print("Note: This reads directly from your friction-penalized vectorbt performance history.")


--- Empirical Risk Engine Update ---
95% 1-Day Historical Sim VaR (%): 0.0000%
95% 1-Day Historical Sim VaR ($): $-0.00
Note: This reads directly from your friction-penalized vectorbt performance history.


In [32]:
import numpy as np
import scipy.stats as stats

# Extract the friction-adjusted daily returns directly from the portfolio simulation
strategy_returns = pf.returns().dropna()

# FIX: Fetch metrics directly using safe, built-in VectorBT properties
# This completely bypasses version-dependent string KeyErrors
ann_return = pf.annualized_return()
max_dd = abs(pf.max_drawdown())

# 1. Sortino Ratio Calculation (Penalizes only downside volatility)
downside_returns = strategy_returns[strategy_returns < 0]
downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
sortino_ratio = ann_return / downside_std if downside_std > 0 else np.nan

# 2. Calmar Ratio Calculation (Return vs Maximum peak-to-trough Drawdown)
calmar_ratio = ann_return / max_dd if max_dd > 0 else np.nan

# 3. 95% 1-Day Historical Simulation Value at Risk (VaR)
# Note: inputting a target value balance of 10000 to match your initial cash deposit
portfolio_value = 10000.0
historical_var_pct = np.percentile(strategy_returns, 5)
historical_var_dollar = -historical_var_pct * portfolio_value

# 4. 95% 1-Day Expected Shortfall / Conditional VaR (Average loss beyond the VaR threshold)
cvar_returns = strategy_returns[strategy_returns <= historical_var_pct]
expected_shortfall_pct = cvar_returns.mean() if len(cvar_returns) > 0 else 0
expected_shortfall_dollar = -expected_shortfall_pct * portfolio_value

# Print final comprehensive system results safely
print("--- Upgraded VectorBT Performance & Risk Report ---")
# If stats_report compiled successfully for general terms, print them:
try:
    print(f"Final Strategy Value Net:  ${stats_report['End Value']:,.2f}")
    print(f"Strategy Win Rate:         {stats_report['Win Rate [%]']:.2f}%")
    print(f"Calculated Sharpe Ratio:   {stats_report['Sharpe Ratio']:.4f}")
except Exception:
    # Fallback to direct properties if stats_report has altered naming structures globally
    print(f"Final Strategy Value Net:  ${pf.final_value():,.2f}")

print(f"Calculated Annualized Return: {ann_return * 100:.2f}%")
print(f"Calculated Max Drawdown:      {max_dd * 100:.2f}%")
print(f"Calculated Sortino Ratio:     {sortino_ratio:.4f}")
print(f"Calculated Calmar Ratio:      {calmar_ratio:.4f}")
print(f"95% 1-Day Hist Sim VaR ($):   ${historical_var_dollar:.2f}")
print(f"95% 1-Day Expected Shortfall ($): ${expected_shortfall_dollar:.2f}")


--- Upgraded VectorBT Performance & Risk Report ---
Final Strategy Value Net:  $10,627.40
Strategy Win Rate:         100.00%
Calculated Sharpe Ratio:   1.5856
Calculated Annualized Return: 7.63%
Calculated Max Drawdown:      0.46%
Calculated Sortino Ratio:     nan
Calculated Calmar Ratio:      16.4314
95% 1-Day Hist Sim VaR ($):   $-0.00
95% 1-Day Expected Shortfall ($): $0.16
